In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf

# Question 1

Consider buying **April S&P E-mini futures** (or the nearest available contract).

- **Entry condition:** A moving average crossover should signal a buy.
  - Fast moving average: **10 minutes**
  - Slow moving average: **20 minutes**
- **Exit strategy:** Once bought, create a stop-loss exit strategy.
  - **Profit-taking price:** 30 ticks above the buying price.
  - **Stop-loss price:** Calculated using the stop-loss model built in class, with a probability of **0.55** for an up move of one tick.

Submit your ADL diagram.

In [8]:
#pull 1-minute ES futures
es = yf.Ticker("ES=F")

# Get the data
es_data = es.history(period="1d", interval="1m")

# Display the data
print(es_data)

                              Open     High      Low    Close  Volume  \
Datetime                                                                
2026-11-06 00:09:00-05:00  7390.25  7390.25  7388.25  7388.25       0   
2026-11-06 00:10:00-05:00  7388.75  7389.25  7387.00  7387.75     305   
2026-11-06 00:11:00-05:00  7387.25  7389.00  7387.25  7387.25     125   
2026-11-06 00:12:00-05:00  7387.25  7387.25  7384.75  7386.00     233   
2026-11-06 00:13:00-05:00  7385.75  7386.00  7382.00  7382.25     353   
...                            ...      ...      ...      ...     ...   
2026-11-06 14:51:00-05:00  7422.25  7425.25  7421.75  7422.25    4085   
2026-11-06 14:52:00-05:00  7422.00  7423.00  7417.75  7422.50    2581   
2026-11-06 14:53:00-05:00  7422.75  7422.75  7420.25  7422.25    1143   
2026-11-06 14:54:00-05:00  7421.75  7422.50  7416.75  7417.00    1763   
2026-11-06 14:55:00-05:00  7417.25  7419.50  7415.75  7418.25       0   

                           Dividends  Stock Splits

In [ ]:
#calculate moving avgs

fast_t = es_data['Close'].rolling(window=10).mean()
slow_t = es_data['Close'].rolling(window=20).mean()

buy_signals = (fast_t > slow_t) & (fast_t.shift(1) <= slow_t.shift(1))

#we want E to represent the first bar with a buy signal, so we will use the index of the first True value in buy_signals
if buy_signals.any():
    entry_time = buy_signals.idxmax()
    E = es_data.loc[entry_time, 'Close']
else:
    print("No buy signals found.")
    E = None

#stop loss size
r = 0.45/0.55
tick_size = 0.25  # ES futures tick size

alpha = 0.05
L = int(np.ceil(np.log(alpha)/np.log(r)))


#check if the exact probability of being stopped out is about alpha
prob_stop = (r**L - r**(L+30)) / (1 - r**(L+30))
prob_target = 1 - prob_stop
print(f"Probability of being stopped out: {prob_stop:.2f}")
print(f"Probability of reaching the target: {prob_target:.2f}")

#calculat exit prices
TP = E + 30 * tick_size
SL = E - L * tick_size

#simulate the trade
exit_price = None
for t, row in es_data.loc[entry_time:].iloc[1:].iterrows():
    if row['Low'] <= SL:
        print(f"Stopped out at {SL} on {row.name}")
        break
    elif row['High'] >= TP:
        print(f"Target reached at {TP} on {row.name}")
        break

#profit and loss
if exit_price is None:
    exit_price, exit_time = es_data['Close'].iloc[-1], es_data.index[-1]
ticks = (exit_price - E) / tick_size
dollars = ticks * 50  # ES futures contract multiplier

#printed summary of ticks, dollars, P(stop), P(target), E, TP, SL
print(f"Entry Price (E): {E}")
print(f"Target Price (TP): {TP}")
print(f"Stop Loss Price (SL): {SL}")
print(f"Ticks: {ticks:.2f}")
print(f"Dollars: {dollars:.2f}")
print(f"Probability of being stopped out: {prob_stop:.2f}")
print(f"Probability of reaching the target: {prob_target:.2f}")

Probability of being stopped out: 0.05
Probability of reaching the target: 0.95
Stopped out at 7385.5 on 2026-11-06 00:41:00-05:00
Entry Price (E): 7389.25
Target Price (TP): 7396.75
Stop Loss Price (SL): 7385.5
Ticks: -9.00
Dollars: -112.50
Probability of being stopped out: 0.05
Probability of reaching the target: 0.95


ADD THE ADL DIAGRAM HERE (SOMEONE GO MAKE IT ON ALGO DESIGN LAB PLS)

# Question 6

Matrix of foreign exchange rates, determining whether there is an arbitrage opportunity in the currency market

In [20]:
#modeling the currency market as a direct weighted graph

nodes = ['EUR', 'GBP', 'CHF', 'USD', 'AUD', 'CAD', 'HKD', 'INR', 'JPY']
lookup = {node: i for i, node in enumerate(nodes)}
matrix = np.ones((len(nodes), len(nodes)))

#fill in the matrix with actual exchange rates given
exchange_rates = {
    ('EUR', 'GBP'): 0.8725,
    ('EUR', 'CHF'): 1.1379,
    ('EUR', 'USD'): 1.145,
    ('EUR', 'AUD'): 1.5745,
    ('EUR', 'CAD'): 1.5052,
    ('EUR', 'HKD'): 8.9846,
    ('EUR', 'INR'): 81.2662,
    ('EUR', 'JPY'): 124.5984,
    ('GBP', 'CHF'): 1.3041,
    ('GBP', 'USD'): 1.3122,
    ('GBP', 'AUD'): 1.8044,
    ('GBP', 'CAD'): 1.725,
    ('GBP', 'HKD'): 10.2969,
    ('GBP', 'INR'): 93.1355,
    ('GBP', 'JPY'): 142.7965,
    ('CHF', 'USD'): 1.0063,
    ('CHF', 'AUD'): 1.3837,
    ('CHF', 'CAD'): 1.3228,
    ('CHF', 'HKD'): 7.896,
    ('CHF', 'INR'): 71.4194,
    ('CHF', 'JPY'): 109.5011,
    ('USD', 'AUD'): 1.3751,
    ('USD', 'CAD'): 1.3146,
    ('USD', 'HKD'): 7.8469,
    ('USD', 'INR'): 70.975,
    ('USD', 'JPY'): 108.8198,
    ('AUD', 'CAD'): 0.956,
    ('AUD', 'HKD'): 5.7064,
    ('AUD', 'INR'): 51.615,
    ('AUD', 'JPY'): 79.1367,
    ('CAD', 'HKD'): 5.9691,
    ('CAD', 'INR'): 53.9906,
    ('CAD', 'JPY'): 82.7791,
    ('HKD', 'INR'): 9.045,
    ('HKD', 'JPY'): 13.868,
    ('INR', 'JPY'): 1.5332  
}

for (from_currency, to_currency), rate in exchange_rates.items():
    i = lookup[from_currency]
    j = lookup[to_currency]
    matrix[i, j] = rate

#fill in lower triangle of the matrix with reciprocal values
for i in range(len(nodes)):
    for j in range(i):
        matrix[i, j] = 1 / matrix[j, i]


df = pd.DataFrame(matrix, index=nodes, columns=nodes)
print(df)


          EUR       GBP       CHF       USD       AUD       CAD        HKD  \
EUR  1.000000  0.872500  1.137900  1.145000  1.574500  1.505200   8.984600   
GBP  1.146132  1.000000  1.304100  1.312200  1.804400  1.725000  10.296900   
CHF  0.878812  0.766812  1.000000  1.006300  1.383700  1.322800   7.896000   
USD  0.873362  0.762079  0.993739  1.000000  1.375100  1.314600   7.846900   
AUD  0.635122  0.554201  0.722700  0.727220  1.000000  0.956000   5.706400   
CAD  0.664364  0.579710  0.755972  0.760688  1.046025  1.000000   5.969100   
HKD  0.111302  0.097117  0.126646  0.127439  0.175242  0.167529   1.000000   
INR  0.012305  0.010737  0.014002  0.014089  0.019374  0.018522   0.110558   
JPY  0.008026  0.007003  0.009132  0.009190  0.012636  0.012080   0.072108   

           INR       JPY  
EUR  81.266200  124.5984  
GBP  93.135500  142.7965  
CHF  71.419400  109.5011  
USD  70.975000  108.8198  
AUD  51.615000   79.1367  
CAD  53.990600   82.7791  
HKD   9.045000   13.8680  
INR

In [30]:
# arbitrage opportunity if there's a negative cycle. building negative log weight matrix

W = -np.log(matrix)  # weight matrix for Bellman-Ford

W_df = pd.DataFrame(W, index=nodes, columns=nodes)
print(df)


          EUR       GBP       CHF       USD       AUD       CAD       HKD  \
EUR -0.000000  0.136393 -0.129184 -0.135405 -0.453938 -0.408926 -2.195512   
GBP -0.136393 -0.000000 -0.265513 -0.271705 -0.590228 -0.545227 -2.331843   
CHF  0.129184  0.265513 -0.000000 -0.006280 -0.324761 -0.279751 -2.066356   
USD  0.135405  0.271705  0.006280 -0.000000 -0.318526 -0.273532 -2.060119   
AUD  0.453938  0.590228  0.324761  0.318526 -0.000000  0.044997 -1.741588   
CAD  0.408926  0.545227  0.279751  0.273532 -0.044997 -0.000000 -1.786596   
HKD  2.195512  2.331843  2.066356  2.060119  1.741588  1.786596 -0.000000   
INR  4.397730  4.534055  4.268570  4.262328  3.943812  3.988810  2.202212   
JPY  4.825096  4.961421  4.695935  4.689693  4.371177  4.416176  2.629584   

          INR       JPY  
EUR -4.397730 -4.825096  
GBP -4.534055 -4.961421  
CHF -4.268570 -4.695935  
USD -4.262328 -4.689693  
AUD -3.943812 -4.371177  
CAD -3.988810 -4.416176  
HKD -2.202212 -2.629584  
INR -0.000000 -0.4273

In [41]:
#running Bellman-Ford algorithm to detect negative cycles

import math


distances = np.full(len(nodes), np.inf)
distances[0] = 0
predecessors = np.full(len(nodes), -1)
#need to account for rounding errors in floating point arithmetic, so we will use a small epsilon value
epsilon = 1e-12

for _ in range(len(nodes) - 1):
    for i in range(len(nodes)):
        for j in range(len(nodes)):
            if distances[i] + W[i, j] < distances[j]:
                if distances[i] + W[i, j] < distances[j] - epsilon:
                    distances[j] = distances[i] + W[i, j]
                    predecessors[j] = i

# 9th pass: if any edge can still be relaxed, a negative cycle exists
found = None
for i in range(len(nodes)):
    for j in range(len(nodes)):
        if distances[i] + W[i, j] < distances[j] - epsilon:
            found = j
            break
    if found is not None:
        break

if found is None:
    print("No negative cycle, no arbitrage.")
else:
    print("Negative cycle detected, arbitrage opportunity exists.")

    # phase 1: step back V times to guarantee we're inside the cycle
    current = found
    for _ in range(len(nodes)):
        current = predecessors[current]

    # phase 2: walk around the cycle once, until we return to start
    start = current
    cycle = [start]
    current = predecessors[start]
    while current != start:
        cycle.append(current)
        current = predecessors[current]
    cycle.append(start)
    cycle.reverse()  # predecessors point backwards, so flip to get trade order

    print("Cycle:", " -> ".join(nodes[k] for k in cycle))

#calculating profit from the arbitrage opportunity
profit_factor = 1.0
for i in range(len(cycle) - 1):
    from_currency = nodes[cycle[i]]
    to_currency = nodes[cycle[i + 1]]
    rate = matrix[cycle[i], cycle[i + 1]]
    profit_factor *= rate
    print(f"{from_currency} -> {to_currency} at rate {rate:.4f}")
print(f"Profit: {profit_factor:.7f}")

#weight cross check
cross_check = 1.0
sum = np.sum(W[cycle[i], cycle[i + 1]] for i in range(len(cycle) - 1))
print(f"Cross-check: {math.e**-sum}")
print(f"Return: {(profit_factor - 1) * 100:.7f} percent")



Negative cycle detected, arbitrage opportunity exists.
Cycle: HKD -> JPY -> INR -> HKD
HKD -> JPY at rate 13.8680
JPY -> INR at rate 0.6522
INR -> HKD at rate 0.1106
Profit: 1.0000149
Cross-check: 1.0000148545615832
Return: 0.0014855 percent


/var/folders/v1/nwsdqxmd3sv3nzy2ybgtmykh0000gn/T/ipykernel_68096/2521420208.py:64: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  sum = np.sum(W[cycle[i], cycle[i + 1]] for i in range(len(cycle) - 1))


### Interpretation

An arbitrage opportunity does exist since we weighted each edge as −ln(rate), turning an arbitrage cycle (a product of rates greater than 1) into a negative-weight cycle. Using Bellman-Ford:

| Leg | Rate |
|---|---|
| HKD → JPY | 13.868 |
| JPY → INR | 1 / 1.5332 = 0.652231 |
| INR → HKD | 1 / 9.045 = 0.110558 |

Profit factor: 13.868 × 0.652231 × 0.110558 ≈ 1.0000149,
which is a return of about 0.0015% per cycle. Starting with 1,000,000 HKD, one
pass through the cycle ends with about 1,000,014.85 HKD, a profit of about 14.85 HKD.

The direct HKD/JPY quote is slightly higher than the rate implied by going through INR . Converting directly gives slightly more yen than the indirect route, so you buy yen directly and
convert back through INR.